# SkyGuard AI — Complete SIH 26073 Project Notebook

**Intelligent real-time anomaly detection for temperature, pressure, and humidity sensors in Automatic Weather Stations**

This teammate-ready notebook explains and verifies the complete project: official data provenance, fault injection, causal features, LightGBM detector, advisory TCN, neighbour-weather gating, hierarchical diagnosis, correction safety, holdout metrics, streaming API, limitations, and reproducible commands.

> Important: the final automatic detector uses only temperature, atmospheric pressure, and relative humidity. Dew point and raw calendar shortcuts are excluded from the compliant Phase 10 model.

## How to use this notebook

1. Open it from the project folder in Jupyter or VS Code.
2. Run cells from top to bottom. The default path uses the already-trained verified artifacts and is laptop-safe.
3. Set `RUN_TESTS = True` near the end to run the automated checks.
4. Keep `RUN_EXPENSIVE_PIPELINE = False` unless you intentionally want to rebuild features and retrain models.
5. Start the API separately with `python src/data/run_api.py`; the live API cell will then verify it.

In [ ]:
from pathlib import Path
import json, sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path.cwd().resolve()
if not (ROOT / 'reports').exists() and (ROOT.parent / 'reports').exists():
    ROOT = ROOT.parent
if not (ROOT / 'reports').exists():
    raise FileNotFoundError('Open this notebook from the Sih 73 project or its notebooks folder.')

REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
FEATURES = ROOT / 'data' / 'features_phase10'
PREDICTIONS = ROOT / 'data' / 'predictions_phase10'
sys.path.insert(0, str(ROOT / 'src'))

def load_json(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))

print('Project root:', ROOT)
print('Python:', sys.version.split()[0], '| OS:', platform.system(), platform.release())

## 1. SIH 26073 requirements and system coverage

| SIH requirement | SkyGuard implementation | Evidence |
|---|---|---|
| Real-time anomaly alerts | Replay/live ingestion, rules, ML probabilities and incident alerts | `streaming_platform.json` and API |
| Spikes, frozen values, sensor faults, communication errors | 13-class fault library plus deterministic packet/order checks | `fault_injection.json` |
| Temporal and seasonal learning | Robust rolling history, EWMA, station climatology, slopes and CUSUM | 108 compliant features |
| Multivariate consistency | Temperature–humidity interaction, pressure relationships and multi-sensor evidence | LightGBM event model |
| Genuine weather versus faults | Neighbour agreement and regional weather gate | Weather decision metrics |
| Confidence and explainability | Calibrated probability, rule evidence, model feature contributions and incident explanation | API/dashboard outputs |
| Sensor degradation and maintenance | Drift/frozen/noise incident diagnosis and sensor-health policy | Health and repair modules |
| Corrected values (optional) | Causal temporal/neighbour estimates with uncertainty and review/automatic safety tiers | Correction reports |
| Fully executable code and examples | Scripts, API, dashboard, tests and this notebook | Project repository |

The ESP32 edge component in the statement is suggested, not compulsory. This software-only system reports CPU latency and model behavior suitable for later edge optimization.

## 2. End-to-end architecture

```text
Official NOAA/NCEI observations
              │
              ▼
Schema + physical + packet quality control
              │
              ▼
Causal temporal features ── Neighbour spatial features ── Seasonal climatology
              │                         │                         │
              └─────────────────────────┴─────────────────────────┘
                                        ▼
                         LightGBM event detector (automatic)
                                        │
                    ┌───────────────────┼────────────────────┐
                    ▼                   ▼                    ▼
          Neighbour-weather gate   TCN sequence evidence   Rule evidence
                    │                (advisory)              │
                    └───────────────────┬────────────────────┘
                                        ▼
                 Incident aggregation + hierarchical root diagnosis
                                        │
                    ┌───────────────────┼────────────────────┐
                    ▼                   ▼                    ▼
             Alert/explanation   Sensor health       Safe correction
                                        │
                                        ▼
                         FastAPI + offline judge dashboard
```

## 3. Validate official data provenance and integrity

The source is NOAA/NCEI Global Hourly (Integrated Surface Database). Validation includes official-source URLs, expected files, SHA-256 checksums, station metadata matching, schemas, station/year coverage, duplicate timestamps, missingness, and physical ranges.

In [ ]:
data_report = load_json('data_validation.json')
summary = data_report['summary']
provenance = data_report['provenance']
data_table = pd.DataFrame({
    'Item': ['Provider', 'Product', 'Processed observations', 'Stations', 'Years',
             'Manifest entries', 'Clean candidate rows', 'Duplicate station timestamps',
             'Ready for anomaly injection'],
    'Value': [provenance['provider'], provenance['product'], f"{summary['processed_rows']:,}",
              summary['stations'], ', '.join(summary['years'].keys()), provenance['manifest_entries'],
              f"{summary['clean_candidate_percent']:.4f}%", summary['duplicate_station_timestamps'],
              data_report['ready_for_anomaly_injection']]
})
display(data_table)
checks = pd.Series(data_report['checks'], name='passed').rename_axis('validation_check').reset_index()
display(checks)
assert data_report['ready_for_anomaly_injection']
assert checks['passed'].all()
print('PASS: all recorded provenance and integrity checks passed.')

In [ ]:
missing = pd.DataFrame(data_report['missing']).T.reset_index(names='parameter')
ranges = pd.DataFrame(data_report['ranges']).T.reset_index(names='parameter')
display(missing.rename(columns={'rows':'missing_rows', 'percent':'missing_percent'}))
display(ranges)
ax = missing.plot.bar(x='parameter', y='percent', legend=False, figsize=(8, 3), color='#2878B5')
ax.set_title('Missing observations by required sensor')
ax.set_ylabel('Percent')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 4. Leakage-safe split and realistic fault library

Rows are not randomly mixed. Training uses 2022, policy/model selection uses 2023, the unseen-time test uses 2024, and a separate 2024 station holdout tests stations never used for training. This is harder and more credible than a random row split.

In [ ]:
fault_report = load_json('fault_injection.json')
split_rows = []
for split, values in fault_report['splits'].items():
    split_rows.append({
        'split': split, 'year': values['year'], 'role': values['evaluation_role'],
        'rows': values['rows'], 'fault_episodes': values['fault_episodes'],
        'weather_episodes': values['weather_episodes'], 'anomaly_rows': values['anomaly_rows']
    })
display(pd.DataFrame(split_rows))
print('Total fault episodes:', fault_report['total_fault_episodes'])
print('Total regional weather episodes:', fault_report['total_weather_episodes'])
print('Total anomalous rows:', f"{fault_report['total_anomaly_rows']:,}")
fault_counts = pd.Series(fault_report['episodes_by_type']).sort_values()
fault_counts.plot.barh(figsize=(9, 5), color='#D9534F', title='Injected fault episodes by type')
plt.xlabel('Episodes'); plt.tight_layout(); plt.show()

## 5. Phase 10 causal feature methodology

The final detector uses 108 features derived only from the three permitted sensor streams and station relationships. Important groups are:

- Basic QC: missing values, gaps, out-of-order indicators and physical consistency.
- Temporal history: lag, rate of change, rolling median/MAD, robust z-score, EWMA residual and frozen-run length.
- Multi-window degradation: 3/6/12/24-hour slopes, neighbour-residual slopes, positive/negative CUSUM and monotonic runs.
- Spatial consistency: neighbour median/mean/MAD, residuals, agreement fractions, distance and observation age.
- Seasonal normality: clean 2022 station month-hour climatology with cluster/global fallback for unseen stations.
- Regional weather evidence: multi-station agreement and standardized disagreement.

All trend windows are causal: they use the current and previously emitted observations only.

In [ ]:
feature_report = load_json('phase10_features.json')
feature_names = feature_report['model_features']
print('Compliant model features:', len(feature_names))
print('Input contract:', feature_report['input_contract'])
print('Dew point used:', feature_report['dew_point_used_by_model'])
print('Causality:', feature_report['causality'])
display(pd.DataFrame({'forbidden_input': feature_report['forbidden_inputs']}))
assert len(feature_names) == 108
assert feature_report['dew_point_used_by_model'] is False
assert not any('dewpoint' in name.lower() for name in feature_names)
assert not any(name in feature_names for name in feature_report['forbidden_inputs'])
print('PASS: the final model obeys the three-parameter SIH input contract.')

In [ ]:
trend_cols = ['station_id','emitted_timestamp_utc','temperature_value','temperature_rolling_median_24h',
              'temperature_slope_6h','temperature_cusum_positive','temperature_cusum_negative','is_anomaly']
trend = pd.read_csv(FEATURES / 'time_test_features.csv.gz', usecols=trend_cols, nrows=12000)
trend['emitted_timestamp_utc'] = pd.to_datetime(trend['emitted_timestamp_utc'], utc=True)
example_station = trend['station_id'].value_counts().index[0]
example = trend[trend['station_id'].eq(example_station)].sort_values('emitted_timestamp_utc').head(500)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(example['emitted_timestamp_utc'], example['temperature_value'], label='Observed temperature', lw=1)
axes[0].plot(example['emitted_timestamp_utc'], example['temperature_rolling_median_24h'], label='24h rolling median', lw=2)
axes[0].set_ylabel('°C'); axes[0].legend(); axes[0].set_title(f'Causal temporal evidence — station {example_station}')
axes[1].plot(example['emitted_timestamp_utc'], example['temperature_slope_6h'], label='6h slope')
axes[1].plot(example['emitted_timestamp_utc'], example['temperature_cusum_positive'], label='CUSUM +', alpha=.8)
axes[1].plot(example['emitted_timestamp_utc'], example['temperature_cusum_negative'], label='CUSUM −', alpha=.8)
axes[1].legend(); axes[1].set_ylabel('Evidence'); plt.tight_layout(); plt.show()

## 6. Models and why each one is used

| Component | Method | Role | Why |
|---|---|---|---|
| Deterministic QC | Physical, missing, duplicate, timestamp and gap rules | Guaranteed handling of known invalid states | Faster and more reliable than asking ML to relearn hard constraints |
| Event detector | Regularized LightGBM gradient-boosted trees | Final automatic anomaly probability | Strong for nonlinear tabular interactions, missing values and CPU deployment |
| Sequence model | Causal dilated Temporal Convolutional Network | Advisory long-pattern evidence | Captures drift/frozen sequences, but was not allowed to increase automatic unseen-station false alarms |
| Weather gate | Neighbour/regional agreement model and policy | Separate regional weather from isolated sensor faults | Directly solves the central SIH example |
| Root diagnosis | Hierarchical rules + multiclass model + incident aggregation | Spike/drift/frozen/noise/etc. classification | Packet faults are deterministic; ambiguous sensor faults need learned evidence and temporal voting |
| Correction | Temporal median, neighbour median/weighted mean, causal ensemble and uncertainty | Advisory repair; limited automatic repair | A safety policy prevents harmful replacements |

### Comparison with traditional AWS quality control

Traditional range/step/persistence checks remain as a safety layer but miss slow drift and context-dependent errors. SkyGuard adds learned normality, multivariate relations, neighbour comparison, calibrated confidence, incident-level diagnosis and explicit separation of genuine regional weather. TabPFN/SAINT/FT-Transformer were not selected because this data is large, time-dependent and deployability-sensitive; they do not automatically solve causal validation or spatial leakage.

## 7. Locked 2024 holdout results

Accuracy alone is misleading because anomalies are rare. The primary measures are precision, recall, F1, AUCPR, false alarms per station-day, episode recall, and weather false-positive rate.

In [ ]:
phase10 = load_json('phase10_final.json')
metric_rows = []
for split in ['time_test', 'station_test']:
    ev = phase10['evaluation'][split]
    binary = ev['binary_fault_detection']
    episode = binary['episode_detection']
    root = ev['end_to_end_root_cause']
    metric_rows.append({
        'split': '2024 unseen time' if split == 'time_test' else '2024 unseen stations',
        'precision_%': 100*binary['precision'], 'recall_%': 100*binary['recall'],
        'f1_%': 100*binary['f1'], 'aucpr_%': 100*binary['aucpr'],
        'false_alarms/station-day': binary['false_alarms_per_station_day'],
        'episode_recall_%': 100*episode['recall'],
        'root_coverage_%': 100*root['diagnostic_coverage'],
        'accepted_root_accuracy_%': 100*root['accepted_root_accuracy'],
        'incident_diagnosis_accuracy_%': 100*root['episode_root']['diagnosed_accuracy']
    })
metrics = pd.DataFrame(metric_rows).set_index('split')
display(metrics)
metrics[['precision_%','recall_%','f1_%','aucpr_%','episode_recall_%']].plot.bar(
    figsize=(11,5), ylim=(0,100), title='Compliant Phase 10 holdout performance')
plt.ylabel('Percent'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
comparison_rows = []
for split, values in phase10['comparison'].items():
    for version in ['phase5_noncompliant', 'phase10_compliant']:
        row = {'split': split, 'model': version}
        row.update({k: 100*v if k != 'false_alarms_per_station_day' else v for k,v in values[version].items()})
        comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows)
display(comparison)
print('Interpretation: Phase 10 is the SIH-compliant model. It slightly improves time-holdout F1, AUCPR, recall and episode recall.')
print('On unseen stations it is deliberately conservative: much higher precision and fewer false alarms, but lower recall/F1.')

In [ ]:
per_fault = phase10['evaluation']['time_test']['binary_fault_detection']['episode_detection']['per_fault_type']
fault_recall = pd.DataFrame(per_fault).T.reset_index(names='fault_type')
fault_recall['recall_percent'] = 100*fault_recall['recall']
display(fault_recall.sort_values('recall_percent'))
fault_recall.sort_values('recall_percent').plot.barh(
    x='fault_type', y='recall_percent', figsize=(9,6), legend=False, color='#F0AD4E',
    title='2024 episode recall by fault type')
plt.xlabel('Detected episodes (%)'); plt.tight_layout(); plt.show()

### Is this best or worst?

It is a **credible working research prototype**, not a finished operational IMD replacement. The strong results are high alert precision, low false-alarm rates, 79.17% episode recall on unseen time, 63.33% on unseen stations, fast CPU processing, and verified real data. The main improvement targets are row-level recall, unseen-station episode recall, root-cause coverage, humidity correction calibration, and evaluation on genuine confirmed hardware-fault logs. Do not present overall event accuracy as the main success metric because normal rows dominate it.

## 8. Independently recompute detection metrics from saved predictions

This cell proves the report values from row-level final predictions rather than merely displaying a stored score.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

recomputed = []
prediction_frames = {}
for split in ['time_test', 'station_test']:
    pred = pd.read_csv(PREDICTIONS / f'{split}_phase10_final_predictions.csv.gz')
    prediction_frames[split] = pred
    y = pred['event_label'].eq('sensor_fault').astype(int) if pred['event_label'].dtype == object else pred['event_label'].astype(int)
    yhat = pred['event_decision'].eq('sensor_fault').astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    recomputed.append({
        'split': split, 'rows': len(pred), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'precision': precision_score(y,yhat,zero_division=0),
        'recall': recall_score(y,yhat,zero_division=0),
        'f1': f1_score(y,yhat,zero_division=0),
        'aucpr': average_precision_score(y,pred['fault_probability'])
    })
recomputed = pd.DataFrame(recomputed).set_index('split')
display(recomputed)
for split in ['time_test','station_test']:
    reported = phase10['evaluation'][split]['binary_fault_detection']
    assert abs(recomputed.loc[split,'f1'] - reported['f1']) < 1e-10
    assert abs(recomputed.loc[split,'aucpr'] - reported['aucpr']) < 1e-10
print('PASS: saved predictions reproduce the locked F1 and AUCPR values exactly.')

In [ ]:
alerts = prediction_frames['time_test'].query("event_decision == 'sensor_fault'").copy()
show_cols = ['station_id','emitted_timestamp_utc','fault_probability','weather_probability',
             'anomaly_type','root_cause_prediction','root_cause_confidence','predicted_incident_id']
display(alerts.sort_values('fault_probability', ascending=False)[show_cols].head(15))

## 9. Load the trained LightGBM bundle and run sample inference

The deployed automatic policy uses LightGBM. The TCN file is retained as advisory sequence evidence because automatic LightGBM+TCN fusion exceeded the unseen-station false-alarm limit. This conservative choice is part of the validated design, not a missing model.

In [ ]:
import joblib
bundle = joblib.load(MODELS / 'phase10_final.joblib')
print('Bundle keys:', list(bundle))
print('Event feature count:', len(bundle['event_features']))
print('Training stations:', len(bundle['training_stations']))
print('TCN role:', bundle['policy']['tcn_role'])
required = ['row_id','station_id','emitted_timestamp_utc'] + bundle['event_features']
sample = pd.read_csv(FEATURES / 'time_test_features.csv.gz', usecols=required, nrows=2000)
X_sample = sample[bundle['event_features']].replace([np.inf,-np.inf], np.nan)
fault_class_index = list(bundle['event_model'].classes_).index('sensor_fault')
sample['live_fault_probability'] = bundle['event_model'].predict_proba(X_sample)[:,fault_class_index]
sample['automatic_threshold'] = bundle['policy']['known_station']['threshold']
sample['automatic_alert'] = sample['live_fault_probability'] >= sample['automatic_threshold']
display(sample[['row_id','station_id','emitted_timestamp_utc','live_fault_probability','automatic_alert']]
        .sort_values('live_fault_probability', ascending=False).head(10))

## 10. Corrected-value estimation and safety policy

Corrections are not blindly substituted. Review suggestions target useful coverage; automatic replacement requires very high precision, agreement among three causal candidates, and a sensor-specific uncertainty limit. Humidity remains review-only because its distribution shift and false-correction risk are larger.

In [ ]:
verification = load_json('final_verification.json')
correction_rows = []
for split_key, split_label in [('correction_time_test','unseen time'), ('correction_unseen_station','unseen stations')]:
    for sensor, values in verification['scorecard'][split_key].items():
        correction_rows.append({
            'split': split_label, 'sensor': sensor, 'coverage_%': 100*values['coverage'],
            'corrected_MAE': values['corrected_value_mae'],
            'MAE_reduction_%': values['mae_reduction_percent'],
            '90%_interval_coverage_%': 100*values['interval_90_coverage']
        })
display(pd.DataFrame(correction_rows))
safe = load_json('safe_repair.json')
safe_rows = []
for split in ['time_test','station_test']:
    for sensor, values in safe['evaluation'][split].items():
        auto = values['auto']
        safe_rows.append({'split':split,'sensor':sensor,'automatic_repairs':auto['proposed_corrections'],
                          'false_repairs':auto['false_positive_corrections'],'precision':auto['precision']})
display(pd.DataFrame(safe_rows))
print('Policy:', safe['policy']['automatic_repair_enabled'])

## 11. Real-time/offline streaming capability

Historical observations are replayed as a live stream for the judge demonstration. The same API contract can accept live sensor or provider records when a network feed is available. Offline replay is essential for a dependable SIH demonstration.

In [ ]:
stream = load_json('streaming_platform.json')
stream_rows = []
for scenario, values in stream['profiles'].items():
    stream_rows.append({'scenario':scenario, 'source_rows':values['source_rows'],
                        'throughput_rows_per_second':values['throughput_rows_per_second'],
                        'mean_latency_ms_per_row':values['mean_processing_latency_ms'],
                        'alerts':values['alerts']})
display(pd.DataFrame(stream_rows))
print('Communication checks:', stream['communication_evidence'])
print('Database:', stream['database'])

In [ ]:
# Optional live API check. Start it in another terminal with: python src/data/run_api.py
import urllib.request
try:
    with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
        live_health = json.loads(response.read().decode('utf-8'))
    print('LIVE API PASS')
    display(live_health)
except Exception as exc:
    print('API is not currently running. This does not affect notebook validation.')
    print('Start it with: python src/data/run_api.py')
    print('Then open: http://127.0.0.1:8000/')

## 12. SIH judging readiness and target levels

There is no official required F1 threshold in the statement. A practical target is to maximize F1/AUCPR while keeping false alarms below about 0.05 per station-day and demonstrating unseen-time and unseen-station generalization. The table below is a planning estimate, **not an official SIH score**.

| Criterion | SIH weight | Current evidence | Main action to increase score |
|---|---:|---|---|
| Innovation & novelty | 25 | Strong: neighbour-weather gate, safe self-healing, dual known/new-station policy | Demonstrate two contrasting judge scenarios |
| Detection accuracy | 20 | Moderate: F1 52.51% time, 47.20% stations; episode recall 79.17%/63.33% | Improve low-recall faults and unseen-station adaptation |
| Real-time capability | 15 | Strong: 5.9k–33.7k rows/s in replay | Show live API and latency counter |
| Explainability | 10 | Good architecture; verify explanations visibly in dashboard | Add concise feature/rule narrative per incident |
| Scalability | 10 | Good CPU throughput and station-independent policy | Benchmark more simultaneous stations |
| Practical deployability | 10 | Strong offline API/database; hardware optional | Package one-command startup and model monitoring |
| Visualization/UI | 5 | Dashboard available | Polish judge flow and incident report |
| Energy efficiency | 5 | CPU-friendly LightGBM; no measured edge power | Report CPU/RAM/model size; optional quantization study |

A defensible internal readiness band is roughly **70–85/100**, depending heavily on demo quality and judge interpretation. It is not honest to claim a guaranteed winning score.

## 13. Reproduce tests or rebuild Phase 10

The test suite is safe to run. Full feature generation and retraining can take substantial time and should only be enabled intentionally. The final required pipeline does not include the experimental weather-gate script because that experiment performed worse and was not promoted.

In [ ]:
import subprocess

RUN_TESTS = False
if RUN_TESTS:
    result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
    assert result.returncode == 0
else:
    print('Tests skipped. Set RUN_TESTS = True and rerun this cell when desired.')

In [ ]:
RUN_EXPENSIVE_PIPELINE = False
pipeline = [
    'src/data/generate_phase10_features.py',
    'src/data/run_phase10_models.py',
    'src/data/train_phase10_full_data.py',
    'src/data/select_phase10_new_station_policy.py',
    'src/data/finalize_phase10.py',
]
if RUN_EXPENSIVE_PIPELINE:
    for script in pipeline:
        print('Running', script)
        subprocess.run([sys.executable, script], cwd=ROOT, check=True)
    print('Phase 10 rebuild complete.')
else:
    print('Expensive rebuild disabled. Set RUN_EXPENSIVE_PIPELINE = True only when intentional.')
    print('Pipeline order:')
    for script in pipeline: print('  ', script)

## 14. Recommended judge demonstration

1. Show all stations healthy and the live throughput/latency counter.
2. Replay a slow pressure drift. Explain the slope, CUSUM and neighbour residual evidence.
3. Show incident aggregation, confidence, root cause, sensor-health change and maintenance recommendation.
4. Show the corrected pressure and uncertainty; explain that correction is advisory unless the strict safety gate passes.
5. Replay a regional temperature rise. Show neighbour agreement and classification as genuine weather instead of a sensor fault.
6. Replay dropout, duplicate packet and timestamp disorder to demonstrate communication handling.
7. Finish with the locked 2024 time/station holdout scorecard—not training accuracy.

## 15. Honest limitations and next improvements

- Labels are realistically injected into genuine weather observations; real confirmed sensor-maintenance labels are still needed before operational claims.
- Row-level recall is moderate, especially for subtle faults and unseen stations. Tune incident-aware objectives and calibrate per climate/sampling tier without touching locked tests.
- Root-cause predictions are accurate when accepted but coverage is limited. Add more causal sequence context and abstain when confidence is low.
- Humidity correction has weaker 2024 uncertainty calibration, so it remains review-only. Use season/sampling-tier conformal calibration and more clean neighbour coverage.
- The TCN helps some sequences but automatic fusion caused excessive unseen-station false alarms. Improve domain adaptation/calibration before promoting it.
- Measure model size, CPU/RAM and actual edge power if pursuing the optional ESP32/Edge-AI marks.

### Final conclusion

SkyGuard AI solves the complete mandatory software scope of SIH 26073 and the optional corrected-value scope, with a genuine official dataset, causal three-parameter detector, spatial weather/fault distinction, explanations, sensor health, API/dashboard, and reproducible holdout metrics. The project is suitable for SIH demonstration now, while the limitations above should be presented honestly as the roadmap from prototype to operational deployment.